In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import os
import glob
from tqdm import tqdm
from PIL import Image
import os
import matplotlib.pyplot as plt

# Load an image
image_path = os.path.join(path,"/kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___Early_blight/009c8c31-f22d-4ffd-8f16-189c6f06c577___RS_Early.B 7885.JPG")
image = Image.open(image_path)

# Display the image
image


# Get all image paths

train_image_paths1 = glob.glob("/kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___Early_blight/*.JPG")


train_image_paths2 = glob.glob("/kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___Late_blight/*.JPG")

train_image_paths3 = glob.glob("/kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___healthy/*.JPG")

train_image_paths  = train_image_paths1+train_image_paths2 +train_image_paths3

test_image_paths1 = glob.glob("/kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___Early_blight/*.JPG")

test_image_paths2 = glob.glob("/kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___Late_blight/*.JPG")

test_image_paths3 = glob.glob("/kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___healthy/*.JPG")

test_image_paths  = test_image_paths1+train_image_paths2 +test_image_paths3


train_labels = []

test_labels = []




print(len(test_image_paths))

for path in tqdm(train_image_paths[0:1599]):
    prefix  = path.split("/")[6].split("_")[3]
    if(prefix=="Early"):
      label=0
    if (prefix=="Late"):
      label = 1
    if (prefix=="healthy"):
      label=2

    train_labels.append(label)

for path in tqdm(test_image_paths[0:1000]):
  prefix  = path.split("/")[6].split("_")[3]
  if(prefix=="Early"):
    label=0
  if (prefix=="Late"):
    label = 1

  if (prefix=="healthy"):
    label=2

  test_labels.append(label)




from torch.utils.data import Dataset
from PIL import Image

class PotatoDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths  # List of image paths
        self.labels = labels  # Corresponding labels
        self.transform = transform  # Transformations to apply

    def __len__(self):
        return len(self.image_paths)  # Total number of images

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]  # Get image path
        label = self.labels[idx]  # Get corresponding label

        # Load image
        image = Image.open(image_path)

        # Apply transformations (if any)
        if self.transform:
            image = self.transform(image)

        return image, label  # Return processed image and its label



from torchvision import transforms
from torch.utils.data import DataLoader

# Define transformations
transformTrain = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),

])

transform_valid_test =transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),

])


train_dataset = PotatoDataset(train_image_paths, train_labels, transform=transformTrain)

test_dataset = PotatoDataset(test_image_paths, test_labels, transform=transform_valid_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)







fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [0,200,500,1000,1345]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]  # Load image & label

    # Convert tensor to numpy for visualization
    img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) → (H, W, C)



    # Show image
    axes[i].imshow(img_np)
    if(label==0):
      class1="Early"
    if(label==1):
      class1="Late"
    if(label==2):
      class1="healty"
    axes[i].set_title(f'Class: {class1}')
    axes[i].axis('off')

plt.show()

In [ ]:
# Write your code here
import torch.nn as nn
class myCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(


            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # [B,32,32,32]
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),  # [B,32,32,32]
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),  # [B,32,32,32]
            nn.ReLU(),
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # [B,32,32,32]
            nn.ReLU(),
            nn.MaxPool2d(2),                             # [B,32,16,16]

            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # [B,64,16,16]
            nn.ReLU(),
            nn.MaxPool2d(2),                            # [B,64,8,8]
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # TO-DO: Calculate input features (channels × height × width)
            # After 2 MaxPool2d(2): 28 → 14 → 7
            nn.Linear(8 * 8 * 64, 3),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)
        return x

In [ ]:
# Write your code here\
import torch
import torch.nn as nn
import torch.optim as optim
 # Training and Validation Loops
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
def validate(model, dataloader, criterion, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total  # Return accuracy



In [ ]:
# Write your code here


from tqdm import tqdm
import torch.optim as optim
model = myCNN()
# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=0.001)

num_epochs = 5

# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, valid_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


'''
# Run Training
model = myCNN()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(1):  # Train for 5 epochs
    train_one_epoch(model, train_loader, criterion, optimizer, device)
    accuracy = validate(model, test_loader, criterion, device)
    print(f"Epoch {epoch+1}: Validation Accuracy = {accuracy:.2f}%")
    '''

In [ ]:
# Write your code here
